#Data Cleaning & Integration 

This notebook handles data loading, cleaning, and preparation of data for the Melbourne Property Intelligence project.


## Note on property types 

This notebook extracts rental data for both 2 bedroom flats and 3 bedroom houses. 
While both property types are retained for descriptive rental market analysis, subsequent rental yield calculations are restricted to house dwellings only, due to the availability of aligned suburb-level price data. 

In [30]:
import pandas as pd 
import numpy as np

In [31]:
#Loading rental dataset from Victorian Rental Report (Master Dataset)
rent_df = pd.read_csv("../data/raw/rtba_moving_annual_median_rent_suburb.csv")
rent_df.head()

,Table 12: Moving annual median rents for suburbs and towns by major property type,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47
0,NaN,NaN,1 Bed Flat,NaN,NaN,NaN,NaN,NaN,NaN,2 Bed Flat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Count,Median,Ann % Ch,Percentile 25,Percentile 75,5 yr % ch,Ave 5 yr % ch,Count,...,Median,Ann % Ch,Percentile 25,Percentile 75,5 yr % ch,Ave 5 yr % ch,Metro/Rural,NaN,NaN,NaN
2,Inner Melbourne,Albert Park-Middle Park-West St Kilda,185,$450,5.1%,$400,$495,26.8%,5.4%,152,...,"$1,500",1.7%,"$1,250","$1,900",7.9%,1.6%,M,NaN,NaN,NaN
3,NaN,Armadale,180,$455,1.1%,$415,$560,23.6%,4.7%,281,...,"$1,638",21.3%,"$1,250","$2,100",48.9%,9.8%,M,NaN,NaN,NaN
4,NaN,Carlton North,37,$425,6.3%,$395,$450,18.1%,3.6%,75,...,-,-,-,-,-,-,M,NaN,NaN,NaN


In [32]:
rent_df.info()
rent_df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 48 columns):
 #   Column                                                                             Non-Null Count  Dtype  
---  ------                                                                             --------------  -----  
 0   Table 12: Moving annual median rents for suburbs and towns by major property type  13 non-null     object 
 1   Unnamed: 1                                                                         159 non-null    object 
 2   Unnamed: 2                                                                         161 non-null    object 
 3   Unnamed: 3                                                                         160 non-null    object 
 4   Unnamed: 4                                                                         160 non-null    object 
 5   Unnamed: 5                                                                         160 non-null    object 

Index(['Table 12: Moving annual median rents for suburbs and towns by major property type',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Contents page',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22',
       'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47'],
      dtype='object')

In [33]:
rent_df.head(20)


,Table 12: Moving annual median rents for suburbs and towns by major property type,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47
0,NaN,NaN,1 Bed Flat,NaN,NaN,NaN,NaN,NaN,NaN,2 Bed Flat,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Count,Median,Ann % Ch,Percentile 25,Percentile 75,5 yr % ch,Ave 5 yr % ch,Count,...,Median,Ann % Ch,Percentile 25,Percentile 75,5 yr % ch,Ave 5 yr % ch,Metro/Rural,NaN,NaN,NaN
2,Inner Melbourne,Albert Park-Middle Park-West St Kilda,185,$450,5.1%,$400,$495,26.8%,5.4%,152,...,"$1,500",1.7%,"$1,250","$1,900",7.9%,1.6%,M,NaN,NaN,NaN
3,NaN,Armadale,180,$455,1.1%,$415,$560,23.6%,4.7%,281,...,"$1,638",21.3%,"$1,250","$2,100",48.9%,9.8%,M,NaN,NaN,NaN
4,NaN,Carlton North,37,$425,6.3%,$395,$450,18.1%,3.6%,75,...,-,-,-,-,-,-,M,NaN,NaN,NaN
5,NaN,Carlton-Parkville,"1,036",$500,6.4%,$435,$570,25.0%,5.0%,"1,193",...,"$1,080",-10.0%,$980,"$1,250",13.7%,2.7%,M,NaN,NaN,NaN
6,NaN,CBD-St Kilda Rd,"6,194",$559,1.6%,$495,$632,31.5%,6.3%,"6,508",...,-,-,-,-,-,-,M,NaN,NaN,NaN
7,NaN,Collingwood-Abbotsford,660,$520,4.0%,$480,$550,24.4%,4.9%,737,...,"$1,150",2.2%,"$1,000","$1,350",40.6%,8.1%,M,NaN,NaN,NaN
8,NaN,Docklands,"1,259",$580,5.5%,$550,$650,26.1%,5.2%,"1,493",...,-,-,-,-,-,-,M,NaN,NaN,NaN
9,NaN,East Melbourne,189,$500,0.0%,$450,$575,16.3%,3.3%,178,...,-,-,-,-,-,-,M,NaN,NaN,NaN


In [34]:
rent_df.tail(20)

,Table 12: Moving annual median rents for suburbs and towns by major property type,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47
143,NaN,Hamilton,25,$250,4.2%,$240,$290,56.3%,11.3%,38,...,$460,2.2%,$420,$480,35.3%,7.1%,R,NaN,NaN,NaN
144,NaN,Horsham,27,$250,-41.9%,$225,$300,2.0%,0.4%,106,...,$520,9.5%,$460,$650,33.3%,6.7%,R,NaN,NaN,NaN
145,NaN,Mildura,79,$280,7.7%,$230,$350,40.0%,8.0%,195,...,$578,9.1%,$500,$625,52.1%,10.4%,R,NaN,NaN,NaN
146,NaN,Moe-Newborough,65,$250,6.4%,$230,$265,56.3%,11.3%,66,...,$545,11.2%,$475,$550,67.7%,13.5%,R,NaN,NaN,NaN
147,NaN,Morwell,43,$265,6.0%,$250,$290,65.6%,13.1%,77,...,$480,1.1%,$440,$550,60.0%,12.0%,R,NaN,NaN,NaN
148,NaN,Ocean Grove-Barwon Heads,-,-,-,-,-,-,-,68,...,$650,7.4%,$590,$700,38.3%,7.7%,R,NaN,NaN,NaN
149,NaN,Portland,16,$300,11.1%,$245,$413,62.2%,12.4%,75,...,$565,13.0%,$480,$650,48.7%,9.7%,R,NaN,NaN,NaN
150,NaN,Sale-Maffra,44,$258,-0.8%,$250,$278,29.0%,5.8%,123,...,$548,7.5%,$500,$570,44.2%,8.8%,R,NaN,NaN,NaN
151,NaN,Seymour,18,$275,-14.1%,$250,$320,52.8%,10.6%,35,...,$520,-1.9%,$500,$550,30.0%,6.0%,R,NaN,NaN,NaN
152,NaN,Shepparton,107,$280,7.7%,$240,$320,43.6%,8.7%,340,...,$580,5.5%,$530,$630,41.5%,8.3%,R,NaN,NaN,NaN


In [35]:
#Skipping header columns
rent_df = pd.read_csv("../data/raw/rtba_moving_annual_median_rent_suburb.csv", skiprows=2)

#Renaming first 2 columns 
rent_df = rent_df.rename(columns={
    rent_df.columns[0]: "region",
    rent_df.columns[1]: "suburb"
})

#Dropping completely empty columns 
rent_df = rent_df.dropna(axis=1, how="all")

#Forward filling region names 
rent_df["region"] = rent_df["region"].ffill()

rent_df.head(20)

,region,suburb,Count,Median,Ann % Ch,Percentile 25,Percentile 75,5 yr % ch,Ave 5 yr % ch,Count.1,...,5 yr % ch.4,Ave 5 yr % ch.4,Count.5,Median.5,Ann % Ch.5,Percentile 25.5,Percentile 75.5,5 yr % ch.5,Ave 5 yr % ch.5,Metro/Rural
0,Inner Melbourne,Albert Park-Middle Park-West St Kilda,185,$450,5.1%,$400,$495,26.8%,5.4%,152,...,11.0%,2.2%,37,"$1,500",1.7%,"$1,250","$1,900",7.9%,1.6%,M
1,Inner Melbourne,Armadale,180,$455,1.1%,$415,$560,23.6%,4.7%,281,...,17.3%,3.5%,18,"$1,638",21.3%,"$1,250","$2,100",48.9%,9.8%,M
2,Inner Melbourne,Carlton North,37,$425,6.3%,$395,$450,18.1%,3.6%,75,...,20.8%,4.2%,-,-,-,-,-,-,-,M
3,Inner Melbourne,Carlton-Parkville,"1,036",$500,6.4%,$435,$570,25.0%,5.0%,"1,193",...,20.0%,4.0%,31,"$1,080",-10.0%,$980,"$1,250",13.7%,2.7%,M
4,Inner Melbourne,CBD-St Kilda Rd,"6,194",$559,1.6%,$495,$632,31.5%,6.3%,"6,508",...,-,-,-,-,-,-,-,-,-,M
5,Inner Melbourne,Collingwood-Abbotsford,660,$520,4.0%,$480,$550,24.4%,4.9%,737,...,18.8%,3.8%,12,"$1,150",2.2%,"$1,000","$1,350",40.6%,8.1%,M
6,Inner Melbourne,Docklands,"1,259",$580,5.5%,$550,$650,26.1%,5.2%,"1,493",...,-,-,-,-,-,-,-,-,-,M
7,Inner Melbourne,East Melbourne,189,$500,0.0%,$450,$575,16.3%,3.3%,178,...,-2.0%,-0.4%,-,-,-,-,-,-,-,M
8,Inner Melbourne,East St Kilda,431,$425,6.5%,$380,$461,28.8%,5.8%,700,...,26.0%,5.2%,30,"$1,125",2.3%,"$1,000","$1,293",25.0%,5.0%,M
9,Inner Melbourne,Elwood,340,$450,7.1%,$400,$500,32.4%,6.5%,632,...,23.0%,4.6%,30,"$1,525",-11.5%,"$1,195","$1,800",29.8%,6.0%,M


In [36]:
rent_df.columns

#from output (mapping is as follows:)
#median = 1 bed flat 
#median.1 = 2 bed flat 
#median.2 = 3 bed flat 
#median.3 = 2 bed house 
#median.4 = 3 bed house 
#median.5 = 4 bed house 

Index(['region', 'suburb', 'Count', 'Median', 'Ann % Ch', 'Percentile 25',
       'Percentile 75', '5 yr % ch', 'Ave 5 yr % ch', 'Count.1', 'Median.1',
       'Ann % Ch.1', 'Percentile 25.1', 'Percentile 75.1', '5 yr % ch.1',
       'Ave 5 yr % ch.1', 'Count.2', 'Median.2', 'Ann % Ch.2',
       'Percentile 25.2', 'Percentile 75.2', '5 yr % ch.2', 'Ave 5 yr % ch.2',
       'Count.3', 'Median.3', 'Ann % Ch.3', 'Percentile 25.3',
       'Percentile 75.3', '5 yr % ch.3', 'Ave 5 yr % ch.3', 'Count.4',
       'Median.4', 'Ann % Ch.4', 'Percentile 25.4', 'Percentile 75.4',
       '5 yr % ch.4', 'Ave 5 yr % ch.4', 'Count.5', 'Median.5', 'Ann % Ch.5',
       'Percentile 25.5', 'Percentile 75.5', '5 yr % ch.5', 'Ave 5 yr % ch.5',
       'Metro/Rural'],
      dtype='object')

In [37]:
#for now, will focus on median.1 (2 bed flat) and median.4 (3 bed house)

two_bed_flat = rent_df[["region", "suburb", "Median.1"]].copy()
two_bed_flat.columns = ["region","suburb", "median_rent"]
two_bed_flat["property_type"] = "2 Bed Flat"

three_bed_house = rent_df[["region","suburb","Median.4"]].copy()
three_bed_house.columns = ["region","suburb","median_rent"]
three_bed_house["property_type"] = "3 Bed House"

rent_clean = pd.concat(
    [two_bed_flat, three_bed_house],
    ignore_index=True
)

In [38]:
rent_clean["median_rent"] = (
    rent_clean["median_rent"].astype(str).str.replace("$","",regex=False).str.replace(",","",regex=False).replace("-",None)
)

rent_clean["median_rent"] = pd.to_numeric(
    rent_clean["median_rent"], errors="coerce"
)

In [39]:
rent_clean.head(50)
rent_clean.tail(50)

,region,suburb,median_rent,property_type
272,South Eastern Melbourne,Springvale,560.0,3 Bed House
273,South Eastern Melbourne,Total,550.0,3 Bed House
274,Mornington Peninsula,Dromana-Portsea,560.0,3 Bed House
275,Mornington Peninsula,Frankston,565.0,3 Bed House
276,Mornington Peninsula,Hastings-Flinders,580.0,3 Bed House
277,Mornington Peninsula,Mt Eliza-Mornington-Mt Martha,675.0,3 Bed House
278,Mornington Peninsula,Seaford-Carrum Downs,550.0,3 Bed House
279,Mornington Peninsula,Total,570.0,3 Bed House
280,Geelong,Belmont-Grovedale,490.0,3 Bed House
281,Geelong,Corio,410.0,3 Bed House


In [40]:
rent_clean.to_csv(
    "../data/processed/clean_rtba_median_rent_suburb.csv",
    index=False
)